# Demo: Baseline Extraction

An agent builds an RDF graph without extra middleware.

---


- **What problem does it solve for a Research Agent?** Frontier models are improving their logical reasoning and mathematical skills all the time.
   - A structured memory is a formal representation of facts, which can **reduce cognitive burden**; e.g., fewer context tokens and fewer agent reasoning cycles.
   - Agents are encouraged to express facts in an _atemporal_ description logic; this avoids the problem of **stale facts accumulating in agent history**. Instead, a bounded description about some facts would automatically describe temporal context and any other necessary metadata.

- **What will this notebook demonstrate?** How to get an agent converting unstructured text into structured knowledge as quickly as possible.
   - What does an agent produce if it doesn't have supporting middleware?
   - How could you integrate this data into a production system?
   - (highly related) How could you build a test harness for metrics?
      - This is fundamental to the questions that really matter: how do we judge and quantify performance?

- **Supporting Middleware?** A set of tools, prompts, and supporting infrastructure for LangChain agents that this repository provides.
   - This demonstration leaves the agent at its own behest with two notable exceptions:
      - Deepagents default middleware is allowed to remain
      - We have a simple `MinistralPromptSuffixMiddleware` that appends a recommended suffix to prompts for Ministral models. This is so that we don't misuse the model and have it impact performance in ways that are unrelated to our task.
   - See [demo-dataset-middleware.ipynb](./demo-dataset-middleware.ipynb) for the next analysis.

## 1. Setup

We've got a few things to do before we get started.

First, we have to filter a LangChain-internal warning (unrelated to our data model, it's just annoying).

In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.",
    category=UserWarning,
)

### 1.1. Bring your own Model

This notebook uses [LangChain's OpenAI integration](https://docs.langchain.com/oss/python/integrations/chat/openai) for [a self-hosted model provider](https://docs.langchain.com/langsmith/playground-model-providers#openai-compatible-endpoint), but you can use any of [LangChain's supported providers](https://docs.langchain.com/oss/python/integrations/providers/all_providers)
What's important is that Deepagents requires a chat model capable of tool invocations.

_NOTE:_ [My Ministral model](https://huggingface.co/mistralai/Ministral-3-14B-Reasoning-2512) recommends adding specific suffix to its prompt. In this repository we also provide `MinistralPromptSuffixMiddleware` to append it so that it doesn't catch people by surprise if they are using other models. Instead, just modify the prompt directly to match your specific model's requirements. Additionally, the model's documentation indicates that one should use `temperature=1` over `temperature=0`, so we follow that guidance as well.

In [ ]:
import os

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("CHAT_MODEL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    seed=42,
    temperature=1,
)

## 2. The Demonstration

We introduce a `SYSTEM_PROMPT` that we be re-used across all of this series demonstration notebooks.

The primary exception is that future prompts will include:

```
You MUST use `add_triples` to add triples to your knowledge base.
You MUST use `serialize_dataset` to serialize your knowledge base before formulating your response.
```

This is meaningless to this demonstration, and would cause the agent to give up upon failing tool calls.

### 2.1. Prompt Setup

- You'll want to set up your prompt for your particular use case.
   - Here, we give the agent the persona of a specialist; the agent should think that it has an analytical mind, which can direct its attention toward skepticism and challenging its first impressions.
- For a resonable baseline, we **have** to do at least _some_ of the prompting that you get "for free" from `DatasetMiddleware`
    - `DatasetMiddleware` automatically educates the agent about their knowledge representation capabilities through `LangChain`'s middleware pattern. Specifically, [we extend the prompt](../rdflib-reasoning-middleware/src/rdflibr/middleware/dataset_middleware.py) to tell the agent about tools and how to use them.
    - Without this, the agent at least to know that it's building a knowledge represntation.
    - Furthermore, we [rigorously annotate our Pydantic types](../rdflib-reasoning-middleware/src/rdflibr/middleware/dataset_model.py), giving agents specification-backed references as well as clean examples. This is hard to give to an agent.

### 2.2. Prompt Limitations

Something important to note here is that the prompt is clearly **trying** to ground the agent in RDF modeling.
The agent is given an explicit set of constraints on what kind of vocabulary terms it should use, but _it will not be able to satisfy those constraints_.
Crucially, the agent has no affordances to expand its context dynamically with background information.
Even if it decides to check its work for compliance, it has no way to conclude that it _is_ in violation of the prompt.

In [ ]:
from typing import Final

SYSTEM_PROMPT: Final[str] = """
You are a research assistant specialized in ontological modeling and the semantic web.
You are given a question to answer or a task to complete.

You have access to tools that allow you to curate a RDF knowledge base, taking advantage of your specialization.
When appropriate, you SHOULD present your answer, reasoning, or supporting evidence in `text/turtle`.

When transforming unstructured text into RDF, you MUST attempt to use controlled RDF vocabularies.
You MUST ensure that your asserted facts are grounded in the provided content; they MUST NOT be assumptions or inferences, unless you have been requested to do so as part of your task.
You MUST NOT assert facts that you are uncertain about given the provided content.
You SHOULD assert all facts that are claimed by the provided content.

Your output MUST include the serialized graph as a `text/turtle` block.
"""

EXTRA_HELP_PROMPT: Final[str] = """
## Knowledge Base

- You have an implicit knowledge base represented as RDF whose content you don't always need to output.
- Use the knowledge base when facts should persist across multiple reasoning steps.
- Use the knowledge base when semantics should be unambiguously represented.
- Use the knowledge base if you are expected to output RDF
   - When presenting RDF to the user or serializing the knowledge base for inspection, you SHOULD prefer Turtle unless the user requests a different RDF serialization.

### Guidance for Modeling Facts

- You MAY incrementally build your dataset using multiple `add_triples` calls.
  - You SHOULD prefer that each `add_triples` call completely describes one single concept or entity.

The following is an example of something that completely describes one single concept or entity.
The subject of this example is a class called `Foo`, but it is analogous to definitions introduced by your knowledge base.

```text/turtle
<urn:ex:Foo> a <rdfs:Class> ;
    rdfs:label "Foo" ;
    rdfs:comment "Foo are known to be used in examples." ;
    rdfs:subClassOf <urn:ex:Bar> .
```

- `<urn:ex:Foo> a <rdfs:Class>` is syntactic sugar for `<urn:ex:Foo> rdf:type <rdfs:Class>`.
- For `rdfs:label`, the object SHOULD usually be a short string literal such as `"Foo"`.
- For `rdfs:comment`, the object SHOULD usually be a descriptive string literal such as `"Foo are known to be used in examples."`.

- Model facts in an atemporal, stable way when possible rather than storing transient phrasing as timeless truth.
- When asserting facts into the knowledge base, you SHOULD keep them grounded in the provided content unless the user explicitly asks for inference, extrapolation, or hypothesis generation.
- You SHOULD NOT assert uncertain facts as settled triples.

### Usage of IRIs and Blank Nodes

- When transforming unstructured content into RDF, you SHOULD prefer controlled vocabularies when they fit the source material and task.
- If you mint IRIs and the user does not specify a base IRI, you SHOULD use <urn:rdflibr:> as the default base for minted IRIs.
- You SHOULD NOT mint IRIs if convention dictates that they be blank nodes (e.g., OWL 2 Class Restrictions).
- You SHOULD prefer a minted IRI over a blank node when there is an authoritative IRI base for that resource.
- When minting an IRI to represent a Class, Datatype, or Property, you MUST assign it a `rdfs:label` and define it using `rdfs:comment`.
"""

### 2.2. Agent Task Definition

This `TASK` will be repeated verbatim across all notebooks in this series.

Here we finally present the agent with a problem: Extract an ontology from this data.

- We are deliberately vague, simply asking if we can formalize the data
   - The given persona tells the agent to focus on the semantic modeling problems that are a throughline of the text.
- We add extraneous information, which we shouldn't see in the output.
- The agent doesn't have the ability to find established vocabularies. Instead it will just make up terms.
   - After all, we're asking it to assert **ALL** facts that are claimed by the content.
   - It's not enough to ask an agent to be grounded; you have to afford it references that it can use to ground itself.

In [ ]:
TASK: Final[str] = """
Please assist me in representing the subject matter of the following text as an RDF graph.
Please use <urn:ex:> as the base for any IRIs that you need to mint as part
of your response. Be sure to label any new terms or properties that you mint
so that they are human readable.

```text
John is a person. Modern people are classified as _homo sapiens_. Apparently,
homo sapiens falls under the subtribe of Hominina. Every Hominidae is a
Haplorhini, and now the names don't even sound like they're related. Then
we get to primates, which can be controversial for some people, for some reason.
Nobody argues with the idea that primates are mammals, yet some people take
umbrage with the idea that _homo sapiens_ is an animal. Oh, Hominidae contains
Hominina, too. Emotions and theological views aside, can we formalize this?
```
"""

#### 2.2.1 Desired Output

We're going to hold the agents to this standard across this series' notebooks.

The agent doesn't have much (any) background knowledge, so we're not going to penalize it for missing best-practices:

1. Concepts that this document introduces sould be documented:
    1.1. They SHOULD have a [`rdfs:label`](https://www.w3.org/TR/rdf-schema/#ch_label)
    1.2. They SHOULD have a [`skos:definition`](https://www.w3.org/TR/skos-reference/#notes)
    1.2. They SHOULD have a description in the form of [`dcterms:description`](https://www.dublincore.org/specifications/dublin-core/dcmi-terms/#description), or at least a [`rdfs:comment`](https://www.w3.org/TR/rdf-schema/#ch_comment)

In [ ]:
from rdflib import Graph

expected_graph = Graph().parse(
    data="""
@prefix ex: <urn:ex:> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

ex:John a ex:Person .

# NOTE: We wouldn't expect owl:equivalentClass unless the agent knows OWL
ex:Person rdfs:subClassOf ex:HomoSapiens ;
          owl:equivalentClass ex:HomoSapiens .

ex:HomoSapiens rdfs:subClassOf ex:Hominina .

ex:Hominina rdfs:subClassOf ex:Hominidae .

ex:Hominidae rdfs:subClassOf ex:Haplorhini .

ex:Haplorhini rdfs:subClassOf ex:Primate .

ex:Primate rdfs:subClassOf ex:Mammal .

ex:Mammal rdfs:subClassOf ex:Animal .

"""
)

print(expected_graph.serialize(format="turtle"))

In [ ]:
from deepagents import create_deep_agent
from rdflibr.middleware.ministral_middleware import MinistralPromptSuffixMiddleware

agent = create_deep_agent(
    model=llm,
    system_prompt=SYSTEM_PROMPT + EXTRA_HELP_PROMPT,
    middleware=[
        MinistralPromptSuffixMiddleware(),
    ],
)

agent_response = agent.invoke({"messages": [{"role": "user", "content": TASK}]})

### 2.4. Manually Assessing Results

This is intersting: When the agent doesn't a generate a fatal syntax error,
the agent can sometimes do a pretty good job.

In [ ]:
from langchain_core.messages.ai import AIMessage

# Get the final AIMessage from the result
message: AIMessage
for message in filter(lambda m: isinstance(m, AIMessage), agent_response["messages"]):
    message.pretty_print()

In [ ]:
import mistune


def extract_turtle_code(markdown_text: str) -> list[str]:
    # 1. Initialize Mistune to return an Abstract Syntax Tree (AST)
    # Using 'ast' is much faster than rendering to HTML
    markdown = mistune.create_markdown(renderer="ast")
    tokens = markdown(markdown_text)

    turtle_blocks = []

    # 2. Iterate through tokens to find fenced code blocks
    for token in tokens:
        # Check if the block is a code block and has the 'turtle' info string
        if token["type"] == "block_code":
            if token.get("attrs", {}).get("info", None) in {"text/turtle", "turtle"}:
                turtle_blocks.append(token["raw"].strip())

    return turtle_blocks


def extract_graphs(markdown_text: str) -> list[Graph]:
    return [
        Graph().parse(data=turtle_block, format="turtle")
        for turtle_block in extract_turtle_code(markdown_text)
    ]


def extract_graph_union(markdown_text: str) -> Graph:
    return sum(extract_graphs(markdown_text), Graph())


try:
    actual_graph = extract_graph_union(message.content)
except Exception as e:
    raise RuntimeError("No valid graph found in final output") from e

### 2.6. Metrics

We have an intersting set of problems in terms of assessing our agent's performance:

1. We are comparing sets (which constrains choices of metrics)
2. If a fact missing from our prediction set, it's _likely_ bad
3. If our prediction set contains additional facts, it's _not necessarily_ bad.

Sections [2.6.1. Why Are Facts Missing?](#261-why-are-facts-missing) and [2.6.2. Extra facts?](#262-extra-facts) will dive into why interpreting the results automatically can be hard.
Section [2.6.3. Are Metrics Insurmountable](#263-are-metrics-insurmountable) will address the problems of _this_ particular result, and how we hope this series of notebooks will address it.

In [ ]:
missing = expected_graph - actual_graph
extra = actual_graph - expected_graph
intersection = expected_graph - missing
union = expected_graph + actual_graph

In [ ]:
import sys

if len(intersection) > 0:
    print(intersection.serialize(format="turtle"))
else:
    print(
        "The agent's graph output did NOT intersect the expected graph.",
        file=sys.stderr,
    )

#### 2.6.1. Why Are Facts Missing?

If a fact is _missing_, then the agent probably worked around it.
It probably made up new prompt-relative RDF Vocabulary terms.

One normally needs to create at least _some_ domain-specific terms when applying semantic web technologies to novel areas, but it's a sledgehammer.
Each new term lacks shared semantics and requires consumers to learn and align with it.
Adding more application-specific terms makes it easier to express your data, but makes it less useful it is to other agents (human or otherwise).d

There's some alternative explanations for missing facts:

1. What if the agent didnt model things explicitly as classes, but as `skos:Concept`s?
Well, that would be _less_ correct, but it wouldn't be impossible to understand.

2. What if the agent uses a more relevent and published domain-specific vocabulary; one that it didn't make up?
In that respect, the agent would be _more_ correct than our labeled examples.
This isn't super likely, beacuse most agents don't have every RDF vocabulary memorized.

> **Requirement:** We need tooling for agents to find and explore new vocabularies.

In [ ]:
print(missing.serialize(format="turtle"))

#### 2.6.2. Extra Facts?

We aren't penalizing the agent for failing to adhere to best practices, but tha doesn't mean that they _won't_.
This may introduce additional documentation triples (`rdfs:label`, `skos:definition`, etc).
In the following output, the agent provides `rdfs:label`s and `rdfs:comment`s on the newly introduced types.

We may also get richer semantics.
[OWL Object Property Restrctions](https://www.w3.org/TR/2012/REC-owl2-syntax-20121211/#Object_Property_Restrictions)
could tell us about relationships that a new class is allowed to have, or certain relationships _exist_ even if they aren't
explicitly represented in the data.
[Shapes Constraint Language](https://www.w3.org/TR/shacl/) could tell us about the graph patterns that a concept is allowed to have, which tells us about graph neighborhoods that are, again, not necessarily in the data.

Agents can't do either of these things in a zero-shot scenario if they aren't aware that they are even things that they can do.

> **Requirement:** Agents needs affordances and the ability to build model patterns.

In the following example, we can also see that the agent inverted the meaning of the `rdfs:subClassOf`.
The agent is aware of the existence of the relationship from its baseline training, but it's clearly not confident as to what it means given `turtle` synatx.

> **Requirement:** Agents need to be able to dive into the meaning of terms so that they can use them appropriately.

In [ ]:
print(extra.serialize(format="turtle"))

#### 2.6.3. Is the Metrics Problem Insurmountable?

I've had some extreme variance in the output of this single prompt:

1. Use of meaningful vocabularies but used incorrectly (e.g., `skos` relationships in violation of domain and range constraints).
2. Use of the expected `rdfs:subClassOf` relationships, including the addition of best practice metadata,.. but the entire class hierarchy was inverted with `rdfs:subClassOf` relationships in the opposite direction.
3. Use of hallucinated output namespaces causing 0% overlap
4. Creation of a RDF Vocabulary specific to this prompt

These kinds of mistakes make **this** baseline useless for a production system with a controlled vocabulary (e.g., some domain-specific terms).
At the very least:

1. There needs to be Boolean quality gates in post-processing of the agent's results
2. For an agent to take corrective action, it needs the ability to check its work.

So metrics will need to be paired with filters and quality gates, and failure to meet those gates are now metrics by themselves.

Next, we need metrics for semantic graphs / set measurements.

- Relatively simple metrics like [Jaccard Index](https://en.wikipedia.org/wiki/Jaccard_index) are hurt by our [Extra Facts Problem](#262-extra-facts).
  If our target graphs are as complete as possible (i.e., as close to a closed world as we can get), then a Jaccard Index is less likely to penalize an overperforming agent.

> **Requirement:** We SHOULD incorporate best-practices triples in our evaluation targets.



- More complex semantic graph metrics typically depend on two graphs with nontrivial node depths / etc.
  Unless we inject the background ontologies defining RDF Vocabularies used in the predicted & target graphs, we are missing most of the Information Content (IC) sources that drive those measures.


In [ ]:
jaccard_index = len(intersection) / len(union)
print(f"The Jaccard Index is {jaccard_index:.3f}")

In [ ]:
from rdflib.extras.external_graph_libs import rdflib_to_networkx_multidigraph

print(f"Converting expected graph to NetworkX ({len(expected_graph)} triples)")
nx_expected = rdflib_to_networkx_multidigraph(expected_graph)

In [ ]:
print(f"Converting actual graph to NetworkX ({len(actual_graph)} triples)")
nx_actual = rdflib_to_networkx_multidigraph(actual_graph)

In [ ]:
from collections.abc import Iterable


def last[T](iterable: Iterable[T]) -> T:
    """Return the last item from an iterable/iterator/generator."""
    for item in iterable:  # noqa: B007
        pass
    return item

In [ ]:
from itertools import islice

import networkx as nx
import tqdm.notebook as tqdm
from pydantic import NonNegativeFloat

iterations = 10
graph_edit_distance: NonNegativeFloat = last(
    tqdm.tqdm(
        islice(nx.optimize_graph_edit_distance(nx_expected, nx_actual), iterations),
        total=iterations,
    )
)


print(f"The Graph Edit Distance is {graph_edit_distance:.3f}")

## 3. Conclusions

> What does this baseline tell us?

A one-shot agent with minimal guidance can produce syntactically valid RDF as turtle, but there are risks if we try to integrate it into a real system.

We've given the agent some pretty strict constraints, but we haven't given it enough background information to work with.
We've allowed it lots of creative flexibility to try to work around it, but that has come at the cost of an unacceptable amount of variance.

> What comes next?

Two primary branches emerge:

1. Give the agent tools to interact with RDF datasets
2. Require the agent to produce structured output

Given agents RDF tools seems fairly intuitive; the agent can avoid syntax errors and work with relatively large knowledge graphs.
We can abstract away any reasoning of the underlying data store as well as complex [SPARQL queries](https://www.w3.org/TR/sparql11-query/).\
We haven't evaded the problem of URI mistakes, an agent can still mint something with the wrong term-separator.

Structured output might _seem_ like abandoning linked data approaches, but its clear once you take [JSON-LD](https://json-ld.org/) into account.
Structured data can have a JSON transformation that is valid JSON-LD, constraining our terms to those that are within our domain or accepted third-party systems.
This is great when you have a strict requirement for your output, but it is extremely constraining when what you want is for the agent to catch unexpected content.

[Our next experiment](./demo-dataset-middleware.ipynb) will explore the approach of providing an agent with tools for interacting with datasets.
We'll identify limitations that continue to exist and possible improvements.